# Aletheia — Neural Relevance Reranker

This Colab notebook reproduces Aletheia's compact deep-learning
experiment for research-paper retrieval. It downloads the pinned
open-access arXiv corpus, extracts and chunks the papers, creates
labelled query–passage candidates from the project benchmark, and
trains a **5 → 16 → 8 → 1** PyTorch MLP to predict relevance.

The comparison is intentionally honest: the held-out split is by
question ID, so passages from a question cannot leak into both
training and validation. The neural model is compared with the
cosine-similarity retrieval baseline using MRR and Recall@6.

**Runtime:** choose a standard CPU or GPU Colab runtime. A GPU is
optional—the MLP is small; downloads and PDF extraction dominate.
Expect roughly 5–15 minutes on a first run, depending on arXiv and
model-cache speed.


## What makes this reproducible

- The benchmark (150 questions over 10 papers) is embedded in this
  notebook, so there is no separate label-file upload.
- The corpus is fetched directly from arXiv and each PDF is checked
  against its pinned SHA-256 hash.
- Positive labels require both the expected paper and a labelled
  physical PDF page; labels are never inferred from the model.
- The deterministic SHA-256 question split keeps evaluation separate
  from training.

This is a feature-based neural reranker, not a transformer fine-tune.
It is deliberately scoped to the model shipped by Aletheia and is
evaluated before any production promotion decision is made.

## Run order and failure recovery

Run the code cells from top to bottom, or use **Runtime → Run all**.
The corpus-download cell must finish with ten `verified` messages before
running extraction. If arXiv is temporarily unavailable, rerun that cell;
already verified PDFs are reused. If you change a data or model cell,
restart the session and rerun from the setup cell so later variables are
not inherited from a partial previous run. Output artifacts are saved in
`/content/aletheia_neural_reranker/outputs`.


In [1]:
# Colab normally provides PyTorch, NumPy, pandas, and scikit-learn.
# Pin the PDF parser because page numbers are part of the gold labels.
!pip -q install pypdf==5.9.0 scikit-learn

import base64
import gzip
import hashlib
import json
import math
import random
import re
import time
from pathlib import Path

from pypdf import PdfReader
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import scipy.sparse as sp
import torch
from sklearn.feature_extraction.text import TfidfVectorizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORKDIR = Path("/content/aletheia_neural_reranker")
PDF_DIR = WORKDIR / "pdfs"
OUTPUT_DIR = WORKDIR / "outputs"
PDF_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Using {DEVICE}; artifacts will be saved to {OUTPUT_DIR}")
import pypdf
print(f"PyTorch {torch.__version__} | pypdf {pypdf.__version__}")


Using cpu; artifacts will be saved to /var/folders/w3/d0mk82h956lg9mz28lwzzrmc0000gn/T/aletheia-notebook-wzz84j04/outputs
PyTorch 2.14.0 | pypdf 5.9.0


In [2]:
# The checked-in Aletheia benchmark is embedded, compressed, below.
# It contains 150 hand-written questions and page-level gold labels.
ENCODED_BENCHMARK = "H4sIAAAAAAAC/8V9eXPbOLbv//MpUH0rFeeOpIhabNmp1Ctnc2c6cdy2M/2qZqZyKQmS2KZINkHaVt+Z99nfWQAQlJUWZdHsqZq0tRE4+AEHZz//+xchfriVqQri6IcT4bXw9VSqSRokGb/3w1WSBlEmDoW89cPcx7fFWEaTxdJPb+A3XZH4CTxC3AtvKH7LpcKvqJZY+NG0fZcGWSYj4c/9IFKZyBZSZPIe/wiUSIJEhkEkhT/Jcj8MVwI+SuGFnIoDJaXgmaiX03yZfJvEaZKrb/jzTrJ60RJRnIlZGi/pqamMpjKFH168+6AEjF18nMpJHIZyQnOPZ/R1nnTnByI59Mcy/JbEYTBZAc3/C+/Bu/I+kTiVb4k/lwrX4iwOp4JeCT+VwmuPfSX5HRHlyzEuQxAVzxeRv4TPxyvxP/T6fzriEw7FP784PXvfDuWtDIEU+E8qJos8uuG3TviFCKb85VTOZSRTH5cGqMCvr+DNJI0nUjG9/INxnEdTPw1gjsv4Voq7IFuItz9+Pf/p2+fT//vt+stP78+vWkLFwtfj3cgVTtJgKu7iPMTXQApAHkxpTKAByaJhB93uy0P4/6jbFepOykRcXL4T/U4PJvRbHqRSdcSFXSVfqMwfh7AkaQxrkK0MBAAUrz+s9TJXGQAcZbBNcKWvZbpU8C0/w2nGKUAIf0UwWgobCz6Jhf42DJWnErYOACDTYALbC3+KIy/9bLLg9bqL06m7NEA+PfyfP/T/+YOYxvAWbhf6Bb45+OcPr0QMs0w3PG8CqLdhO8tIBVlwi4PDlFQ+VhmclTlQ/57WKYadzbQqWEzAw58sYLe34cNgFsCDgAo/SaSf4hRxSczxea5EfBeJebHfxnIWwxTwSwVS8Ew4NfEdPyrNI9FuCzw3GTxGvcR/v9lvfwMgYeJ4duyy84oiOrjoMz+EiS4lvFuajXh7en7+5Rq3BP8AxrPnjk8lLoAv7hYx4AxzoLWUBIv9Ip+IIIO1BxiUCIMbiS/hPMSwaEBBR5yGoUB2oVlKCIuLBwoXEnbPr7AJaOemEhYa+RESS2ug13nZEV+jgqiCHQFmKSASxe6aMpMQ7t5jtO36TGDvz+N0xasDW/DfQFa2iKfwB2zzPKR34D9Bm5jAv8UkjZVqM63/FrkzmbY/hh2TiYMsTmCbBpZV8Aq2mCUq4DmwOSaChgtoW75Yf9I0XuJcD+I8U8FUujjACAGu+4sfgIL/EHeziwBU/IPoYgYHHwVTpAwoiWTW7nqabPiACCg+Kz5wV4RXoPistJuyNJf2EzMF/NUveO5gDdpDIdM0TsU0mBIFl1KdywwoUHKJ4OFpAT5nzsbHJawwfgG3NQCf/Z9i5Aes+h/6E8GXGv/vUP/1L/u7NbZT/OqHfmd49MPm7wMi7m+KT+EDvieuMuKZGl8f+YJP/GvKFyF+ck0bdNjhQf7T+iNgejsCg3tnV1h+jO/EFHl5wIcf/8aFhnGCKVzOcEVlwEVvjBzAfNWCQrTilY7z2xmZox2Q8Ya9RwNzaqBQ/koBiQncjXmCzBOeKkJ/BVynJUb3TH2Kd0Qk/n52JsZ5JpDRwh0dL5NQ3gfZqhJ0/R2hY/byqDN1J4P5IoOpT/wV7bVlvAR2kC8FsmyRo6hytwDuCksQRHBRlY/VMp6CZFINusEOeHU73a7DWuid48cfLS3G9TsDmDoggSSSVFoJjkFzcCxhjUFARIFCBb9LQgTeygI4SyHc+YQACnQOPHDHO3xOw1Q/JL3hYRkPrw48TsTV2buWSzeM0xKfLlEATDO4bzMBQ/FCwOkLbuHuIskSlAh4RBLCYvh5JRyHDXBEQtFyw3nqT1n7AUkEfrlsGUJAhpn4BB+QAiJ3OgOdpzJu3g64ObNw8XPGfDSOH6Msjac5gwlEwYxBGwoyFpNd8uFDFAY3koyyHy5XJOf2yyT7xahDarnfrHEloA8bObAf+aYex1kWgpo1uQGKVTCPQIuFa3HpRyvcw5MbIJgvCXgPpFe4OiRIKcUdOcsjvYKg7Ep/SiTfxdV2wk7iyd4HdtDxTlg3A2phmn1DmTP1Xgukc4TVu/daon/fb+FfqHndxmFOMmUlEI8aESy/gKLy8cPpZdvrtuCi87WkWBYyPa/bJkKNuJkignyWEWv7vV63p79oBB+8+UENrAbmUXE+RzvgetgZ9N2jfdQ5fjzULGIeijmoUQo39+IV2ypGwIpBgwKhRqp1alM46lKhRQjFIudsd8SZjHLQX0GpUwnpiHcxa1KVNsGouavX15pytkjRiLSI02wCAlycsC542hJvCPC3BlMrq9vLF6FNAxQuVP2n93eZxqAjTqfIsR20tY67xt7hlgSlLlvVceTFR/0wcavEhR1OXOk1qgblcSPn+UfDej98+nKh2EiDGPUHequCwACwoQ2OjIhwH/E5prM/QzlfG/xAgm97x3wZ+RX1xv5OymJJnvJGNYhT/RP451CMgzDEN2gNWiAmwenzRs8Kqp4r4R0XX6yCn9dtTtFPJUh1ZMZZwu6+JXGdmCyzX5iLR/yJOA2TdGg0/rdf3n7BCwqNdlM0vKTBZGfVchfu2xvtr1niBiwOXJ/YLhKRoSW0B8htXBE0sgG1ldDzGmGkePrufLV2aZoL1l6Kd36KJu48MUdvpX8F2IMgBTeJn64qXpm7KZVe3SrMoNM7sRoaKipIiApRqYZDl8UxSEjpXJIFEscn2hUKgTmw1LBQq1niCBSZNu/EqPusEq695k7lNCZc7dmaytvA58sRGIsRBFQCb6CxHAUjfxzn2UNJt+LtuMspVEs/DIH8Rx/FyA9XKiBSPhlRj0k50WxHiYUPJ5BdKuh30mM6RJPph2+YSQwAyzRBJbYSkrsafTaYib8DK5no/9is6hmJF+9Lw2WHvXVzqjnJXaH9An8I5Pdx2xkey0JQyzS2xJYzHzR6kzuMFEwQAO4WaCBZs5Yjt3Gogx8Z2a2Nl2QHPjROgju4GoXK05k/kQhprgIUjO1E1h6V+MC0gO5XfEZWoA4tyUmApjWSNFKfPUO4ReBJ0ZR9D1W2xuBP2BpWkDq7+NpexHmq6AZG5R61WBDziXW5a+CYdpvaF9eGfQYZ+jmJGaEgj9JBRAbYOzij7UkYo8KaWmJgRwA0aDnTGp3dP6lMQKqV0474Qnhpr1JpI4l3X/T3HLMoyP2oQQag2pMpBO71kP2f8J2kEs7DfXBm/87jWEAYTGQE+9zoQCBuSPJS69UhM6+2FeM+UFoRhDscHehNoX0e25lqBys7Qjcf9z9c8tv5fLPvCj6o2RroWD7ELAhh5mzbhV1CzAoUzjifL/CmxMmj50ALSqqQj9hkSLyFbh3yZkXMkyaLGFal2p3a2+FODeUse5ki5q7UlCcvp/FdtLfk1ENbUv++b8x9BV24OOir95Ms1xsSfq3VsmJSLaGn0hKwJ2BZtwPee2rALdfUfhVtHjOi04I974olfuMtQ8RhzWbBPDc8DCWN+gH1SuKvd1wDhiD9krPe89ZIhqNoxP1T8pUdu2+9345Vfxes9rD7BFECJy/Ai4u3ngYLUXkLR5ckP7rzpnlKnF6zfMdukNigGrqLiDGFgZzWD2GvN3AxxLgLcXn2pp7TOAvu5bRNiwDj3MP/8dm8NtqM8IBSlY9JhzWuQTMjgVKa3A7zoCGYC6daq/BxEoCuB7TwqlkBB/ZBA860OpybgKDjOiQ/mqUURmiJTyCnSlB0spUYyvZgOzjDhsCZpnFCmiKyP1Kh9QHiMwZYsGDFvOUp3M1DF45ZDupdG74Z0cPrgcbQCGOR4ImHZRakGN94F4u1ITWl2xE6fHqE7J3mytiACgjAk4UitJBX0mlhd3T9APWPuj+VXByDekBRoPwmqDDOUCzDUVwqD44GmsoX25E42gWJx5phVoVFm8PFgB/P4D+OT479FyhYuS43NpXC14BPg8h4dH/EW6x+u/bIq+Ey6p9oOqI4aqP67KNmBp/Oggk7iU3cDcVHFfSAhCGnSoy8Z8D7Ulyl1IdtX+ksjZ4ewXegQoMu6ofWaAQUpiD5Bkr7vtm+iyfquQJunc7w46iqeL+LQwnjLfVwtXiJHgQAnLY/XZ4XcbKGNPj6Kfr8/SJW9G4RK7kesksq+ZzM3tuxO27i9HFoB0MzZlUFT1PbSLVsSeMbC0MVdSQ0UsjmNdKWdeyngl0gnQD5+u3dvUFnVPYLD+uwdxOB7CbeLkJsdBs9BTBJiIY644mnC0n6SxbcjZ/209XfL9+2e11vICahDyKs5SaZr25YGsTI97U4gCd13veiaRmjOqI0hiZ6CiOPULE5EgfPlZzkqeT1gVH1iiHBz1/wpkUVFTVpNLDMQMdRC1k8YgSPgNk9cwMkctICQOC38bdwAR3pqMDn2y9Nz2tIwEz9aE6To1PHQosNnnPPo7XnjVf8XfErJqOkTYTUDb1ePVLNlVA+hjnqiPt/XC2DqCWulv79v3iD4xuvYXR2PML7r2Hs7WjtZD4pItwfLXAWl3ch9nzXaEI2kyImB/MDmB0YRK3wE0k2YleUfQqEhrvYWgaDPWNwevbo9XU0DjzzM+1YJly816E5Q/15r90HDVbe0KGdoZn7/O8f3308FddBBof0TYgi4tnFV9UR72QYjCk1aMfIHNoIO9lmanVUDV1HlQ7RME6qIk5ZakbO6nBJwmrKVv3m4dAoPCykfxvAkk9QHudQSD+KC0+DjYk3Liv0YbBJWHu30BDcAQlKZSlvcY4hgDXBnWGzXLaDOPhTQLQCVBCB4kJXDYbwRpMVkr7EgBS4qGJM9qHlILscrJ09vbB/G/cuoXMHZ26cQ682zJ4BCkgleZQ7whs+GpB9fD8mXiOLM1BLJKY8pijDr9hHBjQknHulgBGHuBpwjaZ5ZJMBcHdWjXyrEZwvOQUHM+kd8SWy3jk+SHAwJrnC84IO+/kKzg5uJaRHZv4sjJOXqj31V8rx2vkCNc32MlCKM0WBa/Ja/iF6xGbwqG92KdmPa3Ys6TsxsBTLGeXF4tkCrCKAE9iMnwYYR6QWwSyr3x5tfYFr8YdoEb5lM8r+5oATPkYLEuL0oXIHxn1gdDBnYDHN2VFd+rH5oitclA37VbHu7Y71nkGNM4mKsZVo4EQmWjTi6+acr5vf+bpBUwjSzwmqJN0DxbBCoAf4k1X9qQbeYO9QuJPyHPkaB30NObDaSH9VtPpNoeXIKo4BgBSsh6HlruJEs2lrkeF3wM64gTnevHJ8xb7JcoPOsXuU0YpQT+4cPPjZ+prA01mlfOVEQPY6RuSln+jv9Z5VxXqwO9Z7KJcoOFNyFqX1P1RZymcTNy4IFpFCyx5yakqhcaOQ2QGJpt76rbOk07ro0r2wv9p5Iub+cukTEWO4X7XyjC9pBO3Ft1ubvIMttIKC5EpJbTKBFyYvD2SOHS7dYXNwO2Id25t19MKbc2RLFlQlUUwswnb1LRTFERuzQbyqhu0uKien+9dgQsCQ1uVSTgNWD50yAi4BLYsn4vdL/tdxVbwOm8MLpFhf52kFIBiNqUTFHMYxea2bLs7CgU/VFlxHCuZhVveb7IIeOc6tNz2OasIxA/l3NpNcF2CCnoQwLEpzYP0JCSsWZWJ9fI5kTOPQ1KPgFWSXv8p1jRI861VhP2qUK2P4PMiIwYQjAq21LypBTQEgRpUjNc+N03TTce3j6rcDJnGSh35NoHsnonjedmpaLGIhvnmECBNzBp0BFwSXFKRHWRXhUaMyVikhWq1nRMP8KXWrBby5fT+kqwj/6nefJLOg6w3LER3d7tH6O4Na/C+dnggBOy5hwl5S43cLV1WROm4UKSvOo9jLIKznUOp8WVJotGwMgsE8QI5rsbQqDQvER7AWz+pH82jdbwbDPBq4D2igRhMu1TM6cWjpe/JQUwwUES2vzOIs/fti0XA+VUXfjf62JwH2LVV8onvhO3oL3rUlcyza8ObLOJiuyRFW8rUk85SqBtDtAu7hcWdUzyk8YcbSvtI0aTO0EjjEM7Iqwd1Z4E1LYBeArIVj6cM5RtvEpCqP9bxGb1E/BaoyIBs3MdUHWatQUea6uB0+n3+8unYsghVFpceV3PG6XfeoHnbryXHvPwjIWgTTqTRhJ9os0e2WzE3M0A67GPyP7je2YBcL1hFXiSmTRX6bMbCxm6q495qxLL77njUJJOZ8orUAqeF+xzFtO1sjdpGLUzkHYSaFvZbWm2hJ1UcCZDqZLkXmDNVCLU/FIA9i4TbANgwASN+Gua4vQVUUH2GFqs31UtzC38O45EkrqUApBg+kqCdEEh5SVHYi10zoR/McdzR5/slJgBEVjflpiswc/UXlTLg8OV320Hxtwyqwz40ZNEAdUPmmMYG/JVXawXnwJ+JMthVU2kC0CEK8edxoB13yEOvggUgShOjpeWwhgxoxfK/nikV+yI8x5TMJv8KM0+y7c0ewUMtlqm0eF2WAdcTpQ+eq3StsYcToFT02LJkn2yMaygcxdEaKIRycEDACjr/ksLUE9lagKuR1OfthuOd+eLSHDz1bGC6gVdxU/sr3WRF+DOo8OrycOqebWMP2Gm01boZLeRvIO7hz0Tq40mleGEW4q1/Vx0KyOP/Nnjn78RNkAHEFjokxGBXVf3Dq1gAMmxJwiTEskFOC+G+K5lX1m38P6zEwnIvX4pCOCVUq2EBBVVR6zaDiluXiZMYpsjdlPTML3FUR0DCDi70NlN1Rirv9kjGBgTDggle/GXDoCHoYItYd1FQYZPqNCX8tYIgWvJzN4G98flWs+rtj9XizrUlFncaYXkVVxhQ7FQrj5W85VTmOY44E+HZDQht/B37X1r8TdpL1m+4oa1FgobMAb8s6jLZ4wKhoRJl2YHWgcsWzDG0DQURGkDnpHaSMYBFsuZSmLEExparoDhrmj/bZGBM1ddxldMZ4r5ar85iyeu6hJDsQPqD+k1iyAR0OaoG2dyIWcOpGTHOL9uxr+PeW/iWiX+IXDgdVYRvuDtsehrw3n95/hROG7iBjpRsH89J15pSM+OXztaBo6/fRPAzUop3F7TOJ+QxPXJGnM6hXVTzseOKljnnvVUXmsDl26XOdsiVqrljlGNVUVnSw3AzIHAprkOeh3JTguCaP1G9yO536y1JAQbe7v5lmiHcaPllHesrM9zjLEf/s4Z+jFkr2KNLDu7wQ39je+hrL0Xc74i0V3qYUKpSOlY3FLEWSwXWDNfmr4n7UKO4LkFPu8Bo0St6mEGjDVck7D+dVa8KlSh71436xZpzzeoIKibjv9TtDgUF4NewH4K0jE/CMQ1Okc4sJh5etLnYeIPhfCjOVFi1Hf+1TM6mqiI8aRZxCDODqS2LsJEAuaRLA0epiL1GXIzu3KBfawuKzmHEVcVagiV2Ry7Gk8oJPUL1QgUbv4j6J19+hLh5JjOG0NVy1wxOMF5ba2EN/oohoVfpZSm7vSYAle/RijCXVp8XlWforWJbM0Rec6eGRwbNFvJaeIitvleNGr2uj8fj3wTJfAjsDThnKaA7/gdXwYfrhrF0IYhzRYstbap+Na0mrHPmwy974cuC9cHfCl4PoxZ65E5jaW6YNB2k5tOAgFUHb6FN7uvPtJMPz/qVOMyDWg56zwEPOlq6HjpdCcK6/vlodtfKGaHC/+Ib0wQ1sqnqvUwcs+Fuo+AtVAfKaOlVU4FCGob1WXT47jVHo1bIuGy6DLKfkADhQlCpXSlGhghooJl/9raI83N0BsWOvs28VYIDLpW9gk65hxjT7lvjgCRyoKlCPMPc8Po+MjeHxxB+jV2dV3JJk8bFBxDLb5Ml8oLI4iugYM0v5N8ay7sYRPWm+aP+oW5aqHq/qXHBmqK70vMokrHKQFstE/P//0YBYuohyR+MbGVF62aAz/Fxki+IPlS0UrZ9oV2nQMreOKT/S7RxW3TKPsDrV5kmB4XTfjs2SlY3Lxg9+CW6CaxAT4L7ob/KI2aZKTVnRL8zsA2n6gXH6kr0xInmnMPYalPQ+1vOkw4AH+1WZmjoM8N7gzwHyD7NJXTCtYsR1Pkuseo0jfMBzvmjMvUnM2ETE97z+Z5cWe/vb6ev9+ICDIcIyjBMqHgxIv8JGVgBE225W+1guX4ryI/nRdJkNWnDLGWgitu8bPD6PVJ7oHWaZYdXdMdxzd+yTAqfFZCMaJmkwIXUBaFrTJxs7ujieVe3RqUlNAUvn7hVNVJfhogIbEaaG/qFDUqbfadmFnzyBkdcIDDp6Jo/QU0t76YH1V7x5f3ndfnN69Z5DNfHVp9PLs/c7FMPbxRVW9rMcHZbMveUqa1537fVhHQkTROqn115P/Pgahhen+OeB53U/v3glmPJPr3sD+BTHx48PxUF/AB9XwLjXHMbrfNVBTq/Fzon2OwlEsCI1NF2zwNDug5tSrn8y7PQqLHx/p4XfQ02EZ2PZS5/rWvwSp9OLQGKAFoto3HtS3bgGQeNT0/YOY0BSRStVoBurgkzrd5SVw6NHZcNgdw9FRd2I//LEZyJVfPp8IrzhM03hK/GPz6dXP/0Lq6kbDy7X6YVrf4otJnGtQK161gLWxCmiU3xZAedBUzjryynCBriFvA3CeDBZK13jfFslIXXJzO4kEPhRneOv8UvncUZ/mxDF+oEe7gtm70QMAS/ek2W6W/SJBs+8WQGsYVNg4c0cp37ZzQJgtdnVgqyxMM+Tj5e+iel+O4fJerug8iaOb9RbkhvK5xCYp3vvtYbdfdhpESQiLgzVZJf2M18cJMMXnFV0dnGtzYyBQj8u8CJvALx2Cmw3wriZcMXtL5R4TmuG29paTThGqSBIHCAZ1K5XcSUjVF4SzF8TB0yR/vB5hb1y+OQ357pdQled1Vfnc+Uwc2uSx830BGUVye+xR/+3JM9efskzrKB7WUrgVCebqFDGpswDa/5bLEUFeI52gmcP2/nZp6/vHVe3I9XEY+r6a3muqaAB4nic6rYt+rbx06ftODvq7lHTrci5J1oxwIQZFj4VrkRfHHWOdBs+UL4xA7LUgOaV9oh7a/1rPC0+VdNGRk0BevVzfvpO3Hra7nzb63Q5I/+DxzgrC7QJYnjSNPvjfqekgoz6e9jYbdjCQeK9KCr0KcHl+vrEY49MqRNqnoTj60XBNTFLgV/HqeiP3FVCUwMuTwVUj5u6cd26zmsJgdSqOwfek3IwNJerdW9mjDNuZ3lE0Sp4CJ6g6WIJ45paMDr5N/1eC9Bl4qhvqhbuqfLbULaHLTGgf/v0bw/+3Q7f5iyymg/lR9PpOtRh93ccz5AkqL9ksVv9VFvjtsrBVKNpCfxpWr8L7OfzTx9dLD/j6/19Yp5TT3N4wtPHPXx+dSEWORZxxIFbAodjRmuObAUcveZwnAHHz0HmG1MKMaCYxto4/jY+//Sp3et2++L8/WXLreXtplYYvDHA07RIQDl55mOi3AKkQ3teyQZJgQzVYD7eydNZaqaJZfTqCFZyUD46QcMILryTUJTFCZfsKyeaYUcqdBPCrJADP1yHCpug18AmIGtQDrCuy0ogHDKs4guc69OPJPtrGf7ql9OzSp2kHnW/jmoppTo42UyOEu+vPn7+6/tPn2MMEOsddTwuVOOQCe/DJCpYE7zdrEa1+bfccE6jf1QL4ORqJ2GzJRVxtlhL2fZWQrUQ2f7Hq/VOX8Yr4jh4OsKokpwhZPuI0XYsyOmIa2pzh+UVkQ/N/HFq6wSnfmKGgk1bAdlB48haw+z1xVcqcLfeuKtsj+Cd3RSGHxwOzvUFtYtJHHCzQI8C8zit8G0Y51Mk48WrYtLuL7XXgwpp3ZP8i5dHPIW9Aa8roDN8PDqPdDhZdCZBZpv96BpCuD055DQFtTnYJsXUebSCcRjE3C2Vbcd6adnAn2xL6Jn636l96Luxv/VYToomZSA8LNFehFlHMx9uLQ6NYp8oxgY/VxzTSdoBhQib8GBuYEFpf/UXRaRyIOv9Xcovjx/f8OXUZiN61CML1uZE+GGy8CloCobWtNJLGxBNL46POSb6NSU6VsC01wCm0lw/JN9j+T+K655g9jWqMfUXKgQKEqrLxf1xTJ2bfQrwfNSF3Oia0LbLIKtamYUXu7/TYj9WaUbhG0vWyXusAATclB/ERdFp5adSBfOIVWQW0PnE+dNbnzxO2Ca1flRg7LPUL5XDv/x8dZHGSS2oULoADkDsQWFInizSpFpCj0Wf6q4jbEKAF4p5NVlrdXpFBTwHjeCpY5MIOczZX3GhLt1Ng6SWmDKky5u9qu+n13qUex3b1e8rflvr1YnQSrEPghvtP91ymcrNwJ2Ao9HFoCtYZTHlb+LbwPAUEBeQf1P6abjSlluK+M58vu4mIXYgwT51FXAdNnPRpXKeygzkmRzzaguMWT4vFYzUzFMvj/kl1q2T6GTK4vp9B18OHu/nW0hQOZZkKrZJ+F8O1G9pdnD94kWJ8FfibZzGITlKsPkIf/8SvvfyGq6xLwfeS/O7CsgdNnIiHT8s8FE/RLUpirFkADc4A7mKkTQAOkYvbrEtEx1firs0oMdRWmf9YTC08VMcuY4Ty30Efdsa++r8kvLeeTW4ECHeu0YqE+Xhub6CSXDx8yzGW3kCCxVJP9xm7WCIjxqEGFD8jJW6tKWKziGgxWkmbIJFoCvaIluPslcFEbZqyVZU2HBv/I60PbIQMnuIJ1cEzAQb97T5pjRyy8rgLIqiINqrANeoGV5qwXquNioOZkM+hT0RFqIGYb9nhP215h325tJKmFAxt9+yrSB0H3KW9nLMTSCDI4iAtrCJem5Woopcc9zUEcPaWW6tMRNtbyrwxnMqYskXhrLVHNZKFFWtZrN3LbLeaH+J51AfwEO466i4Bmakra0DDLQdpc1ZS99F6fFm30Iy0ZljuBmvPnxBNsHJG6Yzn4uKKif2w/fhZE5T/27sT27qd8Mt5RIJ3Zc9HmKWKdIGUx7CEt8LhQ2dU+5VYjpH8oUAm5VHFZhrGKTkM1dCdzPUW7jwT1qMZYUj6HkNgEuqRTBFbFFNKsRJ6qyIhJMR1tfSjL4V3n78cHoJS2OaJ+9cCXAXGz+3itGaTQ3o9k+0add5rFWY4BbUPdEQXyxUFVNNOK5Ioq1JLZR3KCg7w1JRMZaSQl/YvTYwmMiQuSRz24rUzbfn51VA7zXGdymPwLJbOM/GWO7IcW5Z6Larbcp0qep3v6I0qAsN1QD0wEo1OoXaeXxRrJQaDrkOOaR52O1yIHsFxHaz6NTm17GF3B60zPLFpVTwSXvYLddmdVXLpqzO2kTr3M/UkBcVesNFOF6Q57yxQxa3xLo6e8cNe/CRHfEGL86FH97axCcWhWDxWtpNVLidbYNSMohVAHXw54BasnFTvxAy8q41BLbCLbvAbAerwg/WWEm/06KbEjJISohNjS1uPftqI7ZkuJkFlExoDdq2x4S2aY+suZ8rN+KTaTfzI1n9jGJ3Cdr6wZNiubTr79TmOJH2RmWonQKCbrqTnxUNiOiE6ZZR1rPDDJI8+LLCtho+flvtVR0QeBlc6woVOblMwngFLOFdIIERBjfiJ1gKbOuR2eB5s76o+M7iEKSAdp7QXZHkY3QSN+e1Op3NgjAwLclLlQzx1Jv2iZ6u6Bk786W5Tmy7hyUnI6ExUTdv+0PAMJq4dysnm31e5tMn8nuh4hbJO31JuwWaS93AEiwjoJ4iwv+NP2/HszbG95aCya9ugqQNkuKyjuwokB5BXc3jHGu0FuOJg7dvvvzCEY7OV+zQFWHrNQebSS6khsMTAQNgr0af1G6JyqtOxdBucPJNyKUfab2WChJX7Razi5MST7aL3s+5lPv3n/DQ/sc0HuAIL0TbvPzsRy/EX82rX4BlwWvk9WjAUCirCJpDRQj7jUBo3fUWE2pptIK1mOAr812WM7RSd6W/276y38PNC2IMR5ioRZBwUG3lwKedgsJHh8fl9KrDozo6T3gVkRk0i0y2SsghuR0iKyqZtd8tQXEXDGZwmF0MIqx2VAMEThghHLQZ2dIN2bQQIISg47Ign96tiNxwZ+T2zJXK11pxF82atUnd5FY46TLYMywNJpW7Q+wC21kco8p3Lu9KN9vhm9J54ja1NXTrGaA5yRnTrAk6Pyga6vCNzudsbV4B8lia6YBEoGxlq4xTnioCf9gQ8NZeSGzSXNrG3iCD+YIN1ehrwCXEYG+io34pJpRK6SHrabXlPBBhMWTQ9KmdPF7zuLfpi0a9qCqxHDV5NK2K+bZVKiCGA2rCMBlz3R6/u6F9p04fdcBEuwqERrwxsIYu908/Ec9NFDtpCFSHqiAGKKWcEfEWA6e6nefVIBvtDNl+iRRuqxYnuxLwPBx0i7K5tuFxYeWjeHuMW7I3qGkVUcQYcAwpBQvXb8ctVAeHzw6He9avAjWi4DHDYUHerRKoRojeoCXOzz99Fr1+S1zSX8cVz+Nxk+cRDlyaMetwMpyc6lPObYn+ZznxV0/RPKnb6fZKefzszAhX9dyE3yGTRkX/8iSV5H82o5LX0ySacAhQprvqTo11GtSujNOjqgG70W32FKe23JIFkdPWV+tf/xxM0hgrfoNCofOd3saoKtKKvV1giAU2tSbbGtozccWKDU9mXdrUqn6v9nDUOa4B9GE5LaaYvB/GIMUORp1uy4QgTgUO2qIm7XcP0mwrous1hO6Pbv1dJ/a+KLlMDIjyucpikDLwu3Ih9enM6neqTP2Ve5y5m2At9XgHnb65a322+DLBJNb6OEbLhZveLoaviGavQTQ5u4l8KBRlR1Z1GRXNvoJIZdInxoN7VydK61B1R/E3FqDd2rB3mxWVQCTS+RhYimND8rfZpeZObTsap1RVj+PuRpwn6oa1SR8pVffjrEvzFKcWWuNl/XgmgeJCpBMs4WxE8MJzAncCZWo7RG1OfAqqQjX486AC7keaB5yaycKnNJ+0HWIxO4xBwxRh7chDpEBMpy/zMSMBCRnvAlhtSKELySIO4/mqyeZkbJEvcgFAF1w5zWgUVxyM4hI5fP6IlmLWdGPMQn+O/EZXm8lMuAFolTlJ/xT21G5zKJgsdi1W6MASfegyxeZ2XHG3cB5OZRRI1RFvbDFQqqZSFPLQpUGxH56SG2pBFm1bAq5HEKcVt9dwv+1VQ/k/p2sysPA3aRSjooQkKjw2fmikDYCQG2gp03X4Lg2wnl1j2VowN/G1cNvBvK4lthLjHaKLdWpgiu4Gr8zsV8+VS61xgP8hUnP/Oz204IMndHSZTgU6VEEFGFHmRzLOVWj71vlTXAkM5gjFDIvR4f5fj48nZj6N609bmcuI4r7KBl84AJNU94y8lTXUaCmG0dfTGQWEaref5BJLdOy0U7olynPQP3tXJIDoqjXx2PTS89lIhU0vde4VA+GWcdq+S3rN7JLIp+0eFEFKU4n0jqUJ8UhAU6IdbgqhceAVrWOs25UVSxSn9e8NKqMq05kMMndzJDFcBLKmrCYfBC6fAuad0bB2AB6K3PA2HHA7dP2nhu40063mA3hXkFjpBMVhDCS2t5r7WKTvrjBFWhmthBel48MdieyZvfY7F+zZqWJoPU2FdEdwJ3kTjuRvWOsOjfkvew452wEb7ALY/vks1GXID9dg4H6Zs+AejqE9XvVb7BNkP4+vdk7xEMwPPFjz/z64fyFeC3oo/vlSHNi//yqSOfz3xfb1H+6y/nsYitaOwW2QUnXEIiM+xTNPdqGDsxe2hoPPvVHnIfD4kI9XvnRvxoAietG0ZJJXKJJTJRj/WjVtbBcz/t9kBIpAyYYYz8Vg71wygJQofy3a9EAkPZkbgJlk5wsHA0S5J/5b/O3qHQMv/v1v+EUFzA+bOHMrc+SK+8oIQcAmyCGDCWVI67uDs4PfMW3Op0g1isgmqPUXDjzRNl96UX8GmcK6PyhOuKDq8hlpna0ROWgwToN5gBG78fhXk01mZqDzO0EstBZkyqA7w8XEWOzt4B41xFCti+2mtZYIUOat3JMKlbNiI+B7NXjfBs3cfZvKFWDXQ48JB6QUJwlECsGMk619tgmoURNAoZtMFyhStlRrSd+I8JOiNmn9DPPz+cera/doXccpuS4/+JOSxmFCpve3/Yn3xVbaDsRxQycGjYu3rLHO8kinSxXl/IKt4j1hVT9AwItBDi0xv2C+jGGFajDCFuRQnDvlWMwCbNrGaUQU6sTDOQukWmuU0495olsB3egBq1mm+dEJ7QmDGxkGizi2PjsKz6Ir2aiq0w1tYHWuGP8Ik1Zoaz1pht+Fn/5eFl4WwXyhs2Mncv/sP/aHHaJSd+bnSsFjBQ8q7oJoitXwgsz47zHDrDQ+b3mTzmDqrpGZGE4HfdVGAQDrUomPnevEVeJHJoUA3Ymp9G+27xKvIckXBJi2s0XQn/aA/zoly4hdslfxwzvDFMpLWMRG1B9A1iv7x3vd4dG+rey2Q9GMwUWfux9leItxy75QExnBOsbF0aT+idXLMu4SemCHLXeRDEM/gSH3zwg7QXuaeRyI2DEHl+qCUsBjfrfVGdA2du+WZigE85jdhOh6R+N9jooFSqHvtoO4k+mlvkwxYBskcIkrXQbQvUWBJpvOqbSbs+EcMXSfFJMsxThhBibccsAmsXlp2SFReMJI8XtABFr1F/Edtp9JZZE3tL4cXBENO2VkU20xtQkW1Mam3BhrO8qDPwXlcjyNb1Nbc8zZlrNZMAkoYnRzc+h1dtu854zcZVQHlW96M/mi57E5cXxe4U4zLpsH7Zy53mDxDCqRxheq5nBY5HE7ksNHI7mnr+oTbMPrNIa/3qQ+7H1qY4xK/00hBYNcBFsylNP5dn2wZsSogJs5FwDCGDjg8jvzaqGHsqhTcReApLJAnppx9BKJLDDkLJhy+w9zTjWg7M+m3cGiIS5QR1zDouE7ksD3BTbBm6D9i0ooYLoXhrDe6GwbectP/0PIU3++2f0FHzyR+0v3VUp1QgOWIXPeotICgbRuvsvTs116vzom8N5O+fa/9eD/7h1se23UYx23iUQnYp1aMon5RSVQ9AfyfJjy1sY10r+aot3RxN5gp5F7aptkJr8d/F4j4BvlRjeqBpne+K4A4PaVbjdNK4VvXGNywFOY9JYoCkzypXvlkFdTN9Cup1BUiSYSHa2ApbkClujR87A1fe8WcSht6+1XxUKA/BmRXoh421bf9vcJFZq62ZJhRmD3dwF7H+OSITPVQjR1Z7A6P+lyeLIrd395ZLW+dxeXLsxvTi+vayje1+HyfT0sq/GOzt8FXFioY16WyMbh2rodErL9AXbsKdrKbYdr0BRcRWOhab5MHPnDib6eLPKo1HaNFAmzC9GGg7rwU/RYeycnkqzIva43KmcAdtsYTFNSTb3PdZjkS2PSqpDt6VfqGmPG5TWB6x4GLVZiO67DpnBFk0AcETzaBuUcTLiPbaMQYEd5qgsqFJlqfG7xizegPrAntH54YdqmizUshdujAMergSMP2MNiGaaph0LlbujKLJP4ysjkFGJWzM0eatcaW1rF7dAfNgE9qkbBOMWcuoJGLlJBYXSljuAUkv/x4gpuHlRD68f3w+nHqysX1x8DIAI1XhRez/3bYE4WoSssrohReeG0hpSorUgcNYQE7pDrRZzGOQj9wFaKXu0f0ENk3mk5J44jmiIppyQ5LOvHxJVQXGiwuXkNJ264JgOBAIOmW4xZtfWHuI867TnSl+CKSfNIue+TaGOO7XZER7sguofp9j02ahCfuXBd0WELBRrHYntO3tsQk/85T7t+q+xg0CmZZeG1t69Zdg05HKLliKI4BFH383Y8jp8eD3S4xDPMVzAIGBMfFqzh4l06qJ2Y3QNQig5M+lsk7USocsn1O7OQ2Z8AS8/rjOrpooVPelbkAQAxCo23E19JZUJ//ch0ligiKLkIELdN2dIrFuHd6E+rGV6dArrIsTWLju/XcXV/kzGI0tNVEQVuIkkxRHRh9wXiifvi13yKXXCpxt9MN2GlkBK80CvC+bhOzoNe56jUk3uPQ+qmIRUpZoMTbSdxaAPYcWBqEKyhJ9EliFi6wcY+2zH2mjzCxnOKtdI26ckYt+uLeQy79MP7v7+/xPy4YBI+Qd7uUanNxXG3noZLSJelJlAlgjQpCM+R54K2/rm1jOHTvC6+Oq5yWnsNXY7Ih7mKMgqdizhrqzs/Scqp5+Sk0Jkg1pkEW/i5whwPINeYU2X9uYFHpR7dh3UwXRB3jrrP7H1Dyj6uAeiOh8YoF/GrUPqgSQCuh8Cj3S+OSl8cFV/EYx7iatmcWryxloEis28FncPbyepTm3MmhS9GVDKUrxdr9OFmDtrrWJzzhG0oKADOkJEV7LwpMz9Ozk1myrRrDIiJZKCN7qlsI11Y+JeCcdx7NCXMOCNtIoEcUMDa7e/nKvmFgu5kK5HdP9DpHdoB1BGnArOmVdvoosU8Pl45dd6CiBaSVAz0DKW2num7i0vx86lQK5XJpS3nSo4/Mj5PMKxyq4+INtSg6Q1liyKdXXz9bq/TkruAHaQo8i10ynJjziJjQVmAeHLns0MXtZ9UJnGauYGND1p7ocOL9ET1B34+fg4+IuB0qKmE6YTbYRs+GrYa0tCWsGgLrOuOxXxgSy5iZZtEImpTuYyR+f2YzzEHGaMAm3TvOcPyVL5efnIuW0bglZ01UmEbiv2xUzXJ+t/JMINPnqBuWGFgZg5ydnHd7lMm5s4utOHjBF7vaFhPw+eWliSKtCAQEJ4XJkoBI4mxrhJl6aaax6Z0/i2WyojmuT+XxpvGNpY7tEqEIS/Pcxhu+KLliNQ9UHMPktGLCtj2msOWu6fqFKLpN10q07XH+5iwSRfNAi4jpzYcLNUbfRNX9rrsAvrxYcky3+uNRnuaIsiVFn3THWOPD1uWYnp6Cz5jGvkj/HtrCwFGrP/kiBnOx7tLFUW/OFSvekuOR+qcve5gVFebKo3DJLsXrwU+mBOj4OzwTVthwQfNHRGuK0e6JCWcaf7HG2fXgPZdlrzf7RpetP/GdzsghuvZwpyXlsUZ5247A2vqKyAy3AmRPczfVMvR5hpg1WrQ+Oy5sLx8Gdw7sdCmZB4yr7fxcon5X6l/F+5wje1S9ecXOb6G01lqaf8mjm+U9+Cd3uYYEVej7O4Nf++kTPVh91lLmEmKXg9e8fTEyPzZoz8LP+6W9sy8CQ4b2ARUMYh94Nq8UyKNwuG1xkeRhzO8wWdBmFEZkvqP6WB4Xap6OTzqnr2pweUBmOGjOV95iVoeKihJCL8lzq9ptJS1BI3MBFfA6qghrHSk7EzetdUiBo0E1Er6i+DBamT8yvSL1BEp9RvEUQrHyvqc+VEqOhsbBkG2oxrCpAE8QzAy3PhOiZ9EeQLiIFsl6LEEHcbrUk5zt/uCTTgwIZMPaKb0ylk3fiC8fuWsn3432lZEn9Ef7YT+Hua76zS4DfyfT506QsFU36FcOEhrREQJ7w5Wtc36mY1Rf0bS4aDTL9vuOiVb3pHX6dXT6FZ3bwxDrt1VAaDjpgD64DmIOO7GtzGBhjYiNg9T0bGnA2PklX2Po0EZjNGw060FjO2Lvzmz6ymEGWvSwbuMIShESi2+YCP6GGNvp5UTRnaRVf4ObMddZ1tNsYY7rH8i8PFsZaNW9Rip3B4DYXfBFBjdJIRHchMGpnC8Kso5VgDKa0bxuqOof8zULkzsJGjQQixNhxznLi7VgyVvcA6bhEwo2k1RGUsnUnGXkirjvNS2AO3i+J0aBBNxkBy/KAUhD+CtPppX2BRiDd1SwCxsIJ91FaIVGZMkUrgQb1nnmMqMFkBgmTZMLqLn2/6MWqjHU4KWz5n0VTC2vSDYoKaqbJjdbCvclg8xeXQlRJTdsCQg7QNDB590sqBgt598+aBLsXFWwJfxACW6mTjr/cZ3aWsOkR8a9VEyhsIy46/nFUsm9nfhFonM/FkYJy9Vm8pfttyit4M6OsuJ9zR38RWdNx1xkaK8O2XnAwj9fe5rlCxWSgtP5m3tlrh494ELeGGNA8oA9seS9tjAW6/XROpRsaqKOxPZDM8gUzKcVdhVu9l/avOIOfYgK11hSlYOUzj79PW9VX7YoxE06LMonE/MCxw3WKYzUGwCbuFx0UIIOiilf8t1VtAlU1Tr6zjUMV9R3I+Mc16IvVjJkSv7+ag05cwvdPWIQLmtzkQeTaUOmScej00Ng4qiwuDPgd7ovlNq+W2dIGUO0xTabzWPQ+CY85An47ccK57OAgyALbyRWhRGlMSEIonR/FQ4Z7kHFUcUO3V7NGNjejF3CVOo4IhjUWd+CpeLqpybRujtZrVihyl/8VHh06W0TMeZMs6DECW+tVh41/OixQhm81pl4gXl86QFXKkvzdJI1Y1cO90FPa+/d4cPw7+o5Y+kGpo32FaUo64Mlzc8viXCAP1kz52h3VX6x9+vLv7qHf3LLJCtgUMr9RwZAvAg4yV82FYv01wVzl68lh9rNhP8+6+//Ocv/x/aMjTD3hMBAA=="
benchmark = json.loads(gzip.decompress(base64.b64decode(ENCODED_BENCHMARK)))
questions = benchmark["questions"]
answerable_questions = [q for q in questions if q["answerable"]]
print(f"Loaded {len(questions)} questions; {len(answerable_questions)} are answerable.")
print(benchmark["description"])


Loaded 150 questions; 121 are answerable.
Sprint 6 evaluation benchmark: 10 papers x 15 questions, hand-written against the text this pipeline actually extracted (see scripts/dump_corpus_text.py), not from the rendered PDFs and not from recollection of the papers.


In [3]:
# Pinned Aletheia benchmark corpus: ten open-access arXiv PDFs.
# The hashes make accidental corpus drift a hard failure.
PAPERS = [
    ("attention", "1706.03762", "bdfaa68d8984f0dc02beaca527b76f207d99b666d31d1da728ee0728182df697"),
    ("resnet", "1512.03385", "1e0651b6810ecba34a3dbc5b5b0209226f889004607c1f203540a48d64e5a93a"),
    ("bert", "1810.04805", "5692a5514787a8c6727b4ff3b726a3385798bc68e12138d1d4af83947e2acf6e"),
    ("adam", "1412.6980", "eab9c73ae2ceda884b94830bda99312254bac4806f6c9f045cbab90721ecda31"),
    ("vgg", "1409.1556", "83728f9efc21081792902b4c17a4657022d1e4a92ec81655c2427f4ef0755e50"),
    ("batchnorm", "1502.03167", "bdc69e0b568f41d8d3181cc9077a461f58fcdd1c70c9f49efe60a05ee6109655"),
    ("rag", "2005.11401", "23e3249e9a1e75418d82efecab0ea8c4d033b89c93742f63208d47ce01f21233"),
    ("gpt3", "2005.14165", "97fd272f1fdfc18677462d0292f5fbf26ca86b4d1b485c2dba03269b643a0e83"),
    ("word2vec", "1301.3781", "a44d7e22d2005752271c9cc1929c6462d4c8270916b063977992a883e3a54362"),
    ("gan", "1406.2661", "ff5819e3a7b713c3bd3107b7de3d51fe0a347aa5d8444f0efdcf2345ef0a8b63"),
]

def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

session = requests.Session()
session.headers["User-Agent"] = "Aletheia-Colab-Reproducibility-Notebook/1.0"

def download_pdf(arxiv_id, destination, attempts=3):
    url = f"https://arxiv.org/pdf/{arxiv_id}"
    for attempt in range(1, attempts + 1):
        try:
            response = session.get(url, timeout=180)
            response.raise_for_status()
            if not response.content.startswith(b"%PDF"):
                content_type = response.headers.get("content-type", "unknown")
                raise RuntimeError(f"Expected a PDF, received {content_type}")
            destination.write_bytes(response.content)
            return
        except (requests.RequestException, RuntimeError) as error:
            if destination.exists():
                destination.unlink()
            if attempt == attempts:
                raise RuntimeError(
                    f"Could not download arXiv:{arxiv_id} after {attempts} attempts: {error}"
                ) from error
            wait_seconds = 5 * attempt
            print(f"Attempt {attempt}/{attempts} failed for arXiv:{arxiv_id}; retrying in {wait_seconds}s...")
            time.sleep(wait_seconds)

for index, (key, arxiv_id, expected_hash) in enumerate(PAPERS):
    destination = PDF_DIR / f"{key}.pdf"
    if not destination.exists():
        print(f"Downloading {key} (arXiv:{arxiv_id})...")
        download_pdf(arxiv_id, destination)
        # Respect arXiv's guidance for automated clients.
        if index < len(PAPERS) - 1:
            time.sleep(3)
    actual_hash = sha256(destination)
    if actual_hash != expected_hash:
        destination.unlink()
        raise RuntimeError(
            f"SHA-256 mismatch for {destination.name}; the corrupt file was removed. Rerun this cell.\n"
            f"Expected {expected_hash}, got {actual_hash}"
        )
    print(f"✓ {destination.name} verified ({destination.stat().st_size / 1_000_000:.1f} MB)")


✓ attention.pdf verified (2.2 MB)
✓ resnet.pdf verified (0.8 MB)
✓ bert.pdf verified (0.8 MB)
✓ adam.pdf verified (0.6 MB)
✓ vgg.pdf verified (0.2 MB)
✓ batchnorm.pdf verified (0.2 MB)
✓ rag.pdf verified (0.9 MB)
✓ gpt3.pdf verified (6.8 MB)
✓ word2vec.pdf verified (0.2 MB)
✓ gan.pdf verified (0.5 MB)


In [4]:
# Extract page-aware chunks. Keeping page numbers is vital because
# benchmark labels are tied to physical PDF pages, not volatile IDs.
WORD_RE = re.compile(r"[A-Za-z0-9]+(?:\.[0-9]+)?")
HEADING_RE = re.compile(r"^(?:\d+(?:\.\d+)*\s+)?[A-Z][A-Za-z ,:/&()\-]{3,80}$")

def tokens(text):
    return set(WORD_RE.findall(text.lower()))

def likely_heading(line):
    cleaned = " ".join(line.split())
    return bool(HEADING_RE.match(cleaned)) and len(cleaned.split()) <= 12

def make_chunks(key, pdf_path, max_words=300, overlap=40):
    chunks = []
    reader = PdfReader(str(pdf_path))
    active_section = "Unknown"
    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        # Section headings come from the page's own lines; without this the
        # loop below has nothing to iterate and the notebook dies on cell 4.
        lines = [" ".join(line.split()) for line in text.splitlines() if line.strip()]
        for line in lines:
            if likely_heading(line):
                active_section = line
        words = text.split()
        start = 0
        while start < len(words):
            window = words[start : start + max_words]
            content = " ".join(window).strip()
            if len(content) > 80:
                chunks.append({
                    "paper": key,
                    "page": page_number,
                    "section": active_section,
                    "content": content,
                })
            if start + max_words >= len(words):
                break
            start += max_words - overlap
    return chunks

chunks = []
for key, _, _ in PAPERS:
    chunks.extend(make_chunks(key, PDF_DIR / f"{key}.pdf"))

chunks_frame = pd.DataFrame(chunks)
print(f"Created {len(chunks_frame):,} chunks across {chunks_frame.paper.nunique()} papers.")
display(chunks_frame.groupby("paper").size().rename("chunks").to_frame())


Created 486 chunks across 10 papers.
           chunks
paper            
adam           33
attention      27
batchnorm      33
bert           46
gan            21
gpt3          177
rag            43
resnet         42
vgg            36
word2vec       28


In [5]:
# Build retrieval candidates using TF-IDF cosine similarity. The MLP
# never sees the gold page during inference; pages only define labels.
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2, max_features=60_000)
chunk_matrix = vectorizer.fit_transform(chunks_frame.content)
query_matrix = vectorizer.transform([q["question"] for q in answerable_questions])
similarities = (query_matrix @ chunk_matrix.T).toarray()

def candidate_features(query, row, cosine_similarity):
    query_tokens = tokens(query)
    content_tokens = tokens(row.content)
    union = query_tokens | content_tokens
    jaccard = len(query_tokens & content_tokens) / len(union) if union else 0.0
    numbers = {token for token in query_tokens if any(char.isdigit() for char in token)}
    numeric_coverage = len(numbers & content_tokens) / len(numbers) if numbers else 0.0
    section_overlap = len(query_tokens & tokens(row.section)) / len(query_tokens) if query_tokens else 0.0
    content_length_log = math.log1p(len(content_tokens)) / 10.0
    return [cosine_similarity, jaccard, numeric_coverage, section_overlap, content_length_log]

TOP_K = 20
examples = []
for question_index, question in enumerate(answerable_questions):
    top_indexes = np.argsort(similarities[question_index])[-TOP_K:][::-1]
    for chunk_index in top_indexes:
        row = chunks_frame.iloc[chunk_index]
        examples.append({
            "question_id": question["id"],
            "paper": row.paper,
            "page": int(row.page),
            "cosine": float(similarities[question_index, chunk_index]),
            "label": int(row.paper == question["paper"] and int(row.page) in question["expected_pages"]),
            "features": candidate_features(question["question"], row, float(similarities[question_index, chunk_index])),
        })

examples_frame = pd.DataFrame(examples)
print(f"Built {len(examples_frame):,} candidates with {examples_frame.label.sum():,} positives.")
if examples_frame.label.sum() == 0:
    raise RuntimeError("No positive candidates were retrieved. Check corpus extraction or increase TOP_K.")


Built 2,420 candidates with 250 positives.


In [6]:
# Split by question—not individual passages—to prevent leakage.
def is_validation_question(question_id):
    return int(hashlib.sha256(question_id.encode()).hexdigest()[:8], 16) % 5 == 0

examples_frame["validation"] = examples_frame.question_id.map(is_validation_question)
train_frame = examples_frame[~examples_frame.validation].reset_index(drop=True)
validation_frame = examples_frame[examples_frame.validation].reset_index(drop=True)
assert train_frame.label.sum() > 0 and validation_frame.label.sum() > 0

feature_columns = ["cosine", "jaccard", "numeric_coverage", "section_overlap", "content_length_log"]
# Expand named feature columns from the compact list generated above.
examples_frame[feature_columns] = pd.DataFrame(examples_frame.features.tolist(), index=examples_frame.index)
train_frame = examples_frame[~examples_frame.validation].reset_index(drop=True)
validation_frame = examples_frame[examples_frame.validation].reset_index(drop=True)

mean = train_frame[feature_columns].mean().to_numpy(dtype=np.float32)
scale = train_frame[feature_columns].std().to_numpy(dtype=np.float32)
scale[scale < 1e-8] = 1.0
x_train = ((train_frame[feature_columns].to_numpy(np.float32) - mean) / scale)
x_validation = ((validation_frame[feature_columns].to_numpy(np.float32) - mean) / scale)
y_train = train_frame.label.to_numpy(np.float32).reshape(-1, 1)
y_validation = validation_frame.label.to_numpy(np.float32).reshape(-1, 1)

print(f"Train: {len(train_frame)} candidates, {int(y_train.sum())} positive")
print(f"Validation: {len(validation_frame)} candidates, {int(y_validation.sum())} positive")


Train: 1920 candidates, 204 positive
Validation: 500 candidates, 46 positive


In [7]:
# Aletheia's compact neural relevance model: 5 → 16 → 8 → 1.
class NeuralRelevanceMLP(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.network = torch.nn.Sequential(
            torch.nn.Linear(5, 16), torch.nn.ReLU(),
            torch.nn.Linear(16, 8), torch.nn.ReLU(),
            torch.nn.Linear(8, 1),
        )
    def forward(self, x):
        return self.network(x)

model = NeuralRelevanceMLP().to(DEVICE)
x_train_t = torch.tensor(x_train, device=DEVICE)
y_train_t = torch.tensor(y_train, device=DEVICE)
x_validation_t = torch.tensor(x_validation, device=DEVICE)
positive_weight = torch.tensor([(len(y_train) - y_train.sum()) / max(y_train.sum(), 1)], device=DEVICE)
loss_function = torch.nn.BCEWithLogitsLoss(pos_weight=positive_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)

EPOCHS = 300
loss_history = []
for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    loss = loss_function(model(x_train_t), y_train_t)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
    optimizer.step()
    loss_history.append(float(loss.item()))
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch + 1:3d}/{EPOCHS} — weighted BCE: {loss.item():.4f}")

model.eval()
with torch.no_grad():
    neural_scores = torch.sigmoid(model(x_validation_t)).cpu().numpy().ravel()
print("Training complete.")


Epoch  50/300 — weighted BCE: 0.9280
Epoch 100/300 — weighted BCE: 0.8804
Epoch 150/300 — weighted BCE: 0.8460
Epoch 200/300 — weighted BCE: 0.8203
Epoch 250/300 — weighted BCE: 0.7749
Epoch 300/300 — weighted BCE: 0.7231
Training complete.


In [8]:
# Evaluate neural reranking against the original cosine baseline.
def mean_reciprocal_rank(frame, score_column):
    reciprocal_ranks = []
    for _, group in frame.groupby("question_id", sort=False):
        ranked = group.sort_values(score_column, ascending=False)
        positive_positions = np.flatnonzero(ranked.label.to_numpy() == 1)
        reciprocal_ranks.append(1 / (positive_positions[0] + 1) if len(positive_positions) else 0.0)
    return float(np.mean(reciprocal_ranks))

def recall_at_k(frame, score_column, k=6):
    recalls = []
    for _, group in frame.groupby("question_id", sort=False):
        total_positives = int(group.label.sum())
        if total_positives:
            recalls.append(group.nlargest(k, score_column).label.sum() / total_positives)
    return float(np.mean(recalls)) if recalls else 0.0

evaluation = validation_frame.copy()
evaluation["neural"] = neural_scores
cosine_mrr = mean_reciprocal_rank(evaluation, "cosine")
neural_mrr = mean_reciprocal_rank(evaluation, "neural")
cosine_recall = recall_at_k(evaluation, "cosine")
neural_recall = recall_at_k(evaluation, "neural")
accuracy = float(((neural_scores >= 0.5) == y_validation.ravel()).mean())

results = {
    "architecture": "5 -> 16 -> 8 -> 1 (ReLU, ReLU, sigmoid)",
    "feature_names": feature_columns,
    "seed": SEED,
    "candidate_top_k": TOP_K,
    "train_examples": int(len(train_frame)),
    "validation_examples": int(len(validation_frame)),
    "train_positive": int(train_frame.label.sum()),
    "validation_positive": int(validation_frame.label.sum()),
    "final_weighted_bce": loss_history[-1],
    "validation_accuracy": accuracy,
    "cosine_mrr": cosine_mrr,
    "neural_mrr": neural_mrr,
    "mrr_lift": neural_mrr - cosine_mrr,
    "cosine_recall_at_6": cosine_recall,
    "neural_recall_at_6": neural_recall,
    "recall_at_6_lift": neural_recall - cosine_recall,
}
results["recommended_for_future_blended_experiment"] = bool(
    results["mrr_lift"] > 0 and results["recall_at_6_lift"] >= 0
)
display(pd.DataFrame([
    ["MRR", cosine_mrr, neural_mrr, neural_mrr - cosine_mrr],
    ["Recall@6", cosine_recall, neural_recall, neural_recall - cosine_recall],
], columns=["Metric", "Cosine baseline", "Neural MLP", "Lift"]))
print("Decision:", "eligible for a later blended-reranker experiment" if results["recommended_for_future_blended_experiment"] else "not promoted; keep the production reranker unchanged")


     Metric  Cosine baseline  Neural MLP      Lift
0       MRR         0.546000    0.525603 -0.020397
1  Recall@6         0.666667    0.565217 -0.101449
Decision: not promoted; keep the production reranker unchanged


In [9]:
# Save the trained model, normalisation statistics, metrics, and plots.
model_path = OUTPUT_DIR / "neural_relevance_mlp.pt"
torch.save({
    "state_dict": model.state_dict(),
    "architecture": [5, 16, 8, 1],
    "feature_names": feature_columns,
    "normalisation_mean": mean.tolist(),
    "normalisation_scale": scale.tolist(),
    "metrics": results,
}, model_path)
(OUTPUT_DIR / "metrics.json").write_text(json.dumps(results, indent=2) + "\n")
evaluation.drop(columns=["features"], errors="ignore").to_csv(OUTPUT_DIR / "held_out_predictions.csv", index=False)

plt.figure(figsize=(8, 4))
plt.plot(loss_history, color="#2563eb")
plt.title("Neural relevance MLP training loss")
plt.xlabel("Epoch")
plt.ylabel("Weighted binary cross-entropy")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_loss.png", dpi=160)
plt.show()

print("Saved:")
for artifact in sorted(OUTPUT_DIR.iterdir()):
    print(" -", artifact.name)

# Optional: download all reproducibility artifacts to your computer.
# from google.colab import files
# files.download(str(OUTPUT_DIR / "metrics.json"))


Aletheia_Neural_Reranker_Colab.ipynb:cell-9:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
Saved:
 - held_out_predictions.csv
 - metrics.json
 - neural_relevance_mlp.pt
 - training_loss.png


## How to report this experiment

Report the held-out MRR and Recall@6 against the cosine baseline,
state the question-level split and the class imbalance, and retain
`metrics.json`, `held_out_predictions.csv`, and the model checkpoint
as your reproducibility artifacts. A gain here means the MLP is a
candidate for a later blended-reranker experiment—it does **not**
automatically replace Aletheia's production cross-encoder.

## Why this notebook's numbers differ from the shipped model

Run this end to end and the MLP comes out *below* the baseline it is compared
against — around -0.02 MRR and -0.10 Recall@6 on the held-out split. The model
shipped in `backend/models/neural_relevance_model.json` reports +0.173 MRR and
+0.153 Recall@6 against its own baseline. Both numbers are real; they are not
the same experiment.

The difference is the candidate set. This notebook retrieves with TF-IDF
cosine similarity so that it can run anywhere with no model download. The
shipped pipeline retrieves with `jina-embeddings-v2-small-en` and chunks
through the production parser, so its candidates, its `cosine` feature and its
baseline are all different. A reranker's gain is measured against whatever
retrieved for it, and a stronger retriever leaves less to recover.

Read this notebook as a reproducible demonstration of the architecture, the
labelling rule and the leakage-free split — not as a reproduction of the
shipped metrics. Reproducing those needs the ingested corpus and
`backend/scripts/train_neural_reranker.py`.
